# 03 · Analysis: RecGym sensor-efficiency study

Reads the outputs of notebook 01 (RF, SVM, XGBoost) and notebook 02 (1D CNN) and produces every table and figure for the paper. **It trains nothing.** Every number is computed from `results.csv` and `confusion_long.csv`, and every table is saved as CSV.

**Inputs (upload or keep on Drive):** `recgym_01_classical_full.zip`, `recgym_02_cnn_full.zip`.
**Output:** `recgym_03_analysis.zip` (tables, figures as PNG and PDF, integrity checks, manifest, log). Upload that zip back.

## Frozen analysis plan (fixed in this notebook before any test was run)

Descriptive means from notebooks 01 and 02 had already been seen. The inferential plan below was fixed before any test was computed.

| Block | Question | Unit of analysis | Test family for Holm correction |
|---|---|---|---|
| A | Cost of channel reduction: each of the 14 reduced configs vs 6-axis | per-subject (LOSO fold) macro-F1, paired | position × model (14 tests) |
| B1 | RQ3: accelerometer-minus-gyroscope gap at three levels (triples, pairs, singles) | paired differences per subject | model (3 positions × 3 levels = 9) |
| B2 | RQ3: does that gap differ between positions? | difference of gaps per subject (same subjects at every position) | model (3 position pairs × 3 levels = 9) |
| C | RQ3: marginal gain of each axis given a non-empty subset of the same sensor's other axes | per-subject mean gain | model (3 positions × 6 axes = 18) |
| D | Model comparison (6 model pairs) on 6-axis, acc-3, gyro-3, and pooled over all 15 configs | paired per subject | position × config (6 tests) |
| E | Position comparison (3 position pairs) | paired per subject (same person at each position) | model × config (3 tests) |
| F | RQ2: performance vs cost (channels, data rate, size, latency), Pareto flags, descriptive non-inferiority at margins 0.02, 0.03 and 0.05 | descriptive | none |
| G | RQ4: per-class F1 drop for acc-3 and gyro-3 vs 6-axis, pooled confusions | per class, only folds where the class is present | position × model (24 tests) |
| H | Per-subject results | descriptive | none |

**Primary metric:** `macro_f1` (all 12 labels, over classes present in the held-out subject). **Null-sensitivity metric:** `macro_f1_workouts` (11 workouts, Null excluded). Blocks A to E are repeated on both metrics.

**Statistics:** two-sided Wilcoxon signed-rank on the 10 paired fold values, Holm correction within the families above, plus a stricter Holm over every test in the table (`p_holm_global`). Effect sizes: mean paired difference with a percentile bootstrap CI over subjects (crude with n = 10), Cohen's dz, matched-pairs rank-biserial correlation, and sign counts. With n = 10, the smallest possible exact two-sided p is 2/1024 ≈ 0.00195, so a global Holm over more than about 25 tests can never reject. That is a property of the design, not a bug. Read the effect sizes and sign counts, not only the p-values.

**Interpretation guardrails (from the integrity rules):**
- Axes are in the **device frame**. Axis results are placement- and orientation-specific and must not be generalized to body axes.
- Do not name a "best configuration" on tiny gaps.
- CNN results come from **one seed** unless more seeds were added. Training noise is then not quantified.
- Latency is single-window CPU latency on Colab hardware. Notebook 01 did not record its CPU model, so classical-versus-CNN latency comparisons are indicative only.
- The dataset was normalized by its providers with an undocumented method.

In [ ]:
# =========================== PARAMETERS (edit here only) ===========================
RESULTS_DIR = os.environ.get("RESULTS_DIR", "results/analysis")   # where this notebook writes tables/figures/checks
INPUT_DIR = os.environ.get("INPUT_DIR", "results/raw")   # where the notebook 01/02 outputs are read from
ZIP_01 = os.environ.get("ZIP_01", "recgym_01_classical_full.zip")
ZIP_02 = os.environ.get("ZIP_02", "recgym_02_cnn_full.zip")

SEED = 42                 # bootstrap seed (the same resamples are reused for every comparison)
N_BOOT = 10000
ALPHA = 0.05
DELTAS = [0.02, 0.03, 0.05]          # non-inferiority margins in macro-F1 (descriptive, declared before looking at CIs)
PRIMARY_METRIC = "macro_f1"          # 12 labels, classes present in the held-out subject
SENS_METRIC = "macro_f1_workouts"    # 11 workouts, Null excluded
METRICS = [PRIMARY_METRIC, SENS_METRIC]
FIG_MODELS = ["XGB", "CNN"]          # models drawn in the per-class and confusion figures (tables cover all four)
KEY_CONFIGS = ["6-axis", "acc-3", "gyro-3"]
FS = 20
DPI = 200

In [ ]:
import os, sys, json, re, zipfile, shutil, hashlib, itertools, platform, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd
import scipy, matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import wilcoxon, rankdata, kendalltau, spearmanr
from IPython.display import display
warnings.filterwarnings("ignore", category=UserWarning)

IN_COLAB = "google.colab" in sys.modules
OUT = Path("/content/outputs/03_analysis") if IN_COLAB else Path(RESULTS_DIR)
IN_DIR = Path("/content/analysis_inputs") if IN_COLAB else Path(INPUT_DIR)
for sub in ["tables", "figures", "checks"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted:", e)

def find_input(name):
    cands = [Path(INPUT_DIR) / name, Path(name), Path("/content") / name, Path("/mnt/user-data/uploads") / name]
    dr = Path(INPUT_DIR)
    if dr.exists():
        cands += list(dr.rglob(name))
    for c in cands:
        if c.exists():
            return c
    return None

zip1, zip2 = find_input(ZIP_01), find_input(ZIP_02)
missing = [n for n, z in [(ZIP_01, zip1), (ZIP_02, zip2)] if z is None]
if missing and IN_COLAB:
    from google.colab import files
    print("Please upload:", missing)
    files.upload()
    zip1, zip2 = find_input(ZIP_01), find_input(ZIP_02)
assert zip1 is not None and zip2 is not None, f"Input zips not found: {ZIP_01}, {ZIP_02}"

def unpack(zp, tag):
    d = IN_DIR / tag
    if d.exists():
        shutil.rmtree(d)
    with zipfile.ZipFile(zp) as z:
        z.extractall(d)
    return d

def locate(root, fname):
    hits = list(Path(root).rglob(fname))
    assert len(hits) == 1, f"Expected exactly one {fname} under {root}, found {hits}"
    return hits[0]

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

d1, d2 = unpack(zip1, "cls"), unpack(zip2, "cnn")
log("Inputs:", zip1, zip2)
log("Versions: python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
    "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)

def save_table(t, name):
    p = OUT / "tables" / f"{name}.csv"
    t.to_csv(p, index=False)
    log("saved table:", p.name, t.shape)
    return t

def savefig(fig, name):
    fig.savefig(OUT / "figures" / f"{name}.png", dpi=DPI, bbox_inches="tight")
    fig.savefig(OUT / "figures" / f"{name}.pdf", bbox_inches="tight")
    log("saved figure:", name)
    plt.show()
    plt.close(fig)

## 1 · Load results and run integrity checks

Every check below must pass. A failure stops the notebook so that nothing downstream is built on incomplete or inconsistent inputs.

In [ ]:
POS = ["leg", "pocket", "wrist"]
MODELS = ["RF", "SVM", "XGB", "CNN"]
CFG_ORDER = ["6-axis", "acc-3", "gyro-3", "acc-AxAy", "acc-AxAz", "acc-AyAz",
             "gyro-GxGy", "gyro-GxGz", "gyro-GyGz",
             "single-Ax", "single-Ay", "single-Az", "single-Gx", "single-Gy", "single-Gz"]
IMU = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
CLASSES = ["Adductor", "ArmCurl", "BenchPress", "LegCurl", "LegPress", "Null", "Riding",
           "RopeSkipping", "Running", "Squat", "StairClimber", "Walking"]
WORKOUTS = [c for c in CLASSES if c != "Null"]
F1COLS = [f"f1_{c}" for c in CLASSES]
COL = {"RF": "#1b9e77", "SVM": "#d95f02", "XGB": "#7570b3", "CNN": "#e7298a"}

res_cls = pd.read_csv(locate(d1, "results.csv"))
res_cnn = pd.read_csv(locate(d2, "results.csv"))
conf_cls = pd.read_csv(locate(d1, "confusion_long.csv"))
conf_cnn = pd.read_csv(locate(d2, "confusion_long.csv"))
sanity = json.load(open(locate(d2, "sanity_checks.json")))
env_cnn = json.load(open(locate(d2, "environment.json")))
env_cls = json.load(open(locate(d1, "environment.json")))
cfg_cnn = json.load(open(locate(d2, "run_config.json")))
cfg_cls = json.load(open(locate(d1, "run_config.json")))

CHECKS = {}
def check(name, ok, detail=""):
    CHECKS[name] = {"ok": bool(ok), "detail": str(detail)}
    log(("PASS  " if ok else "FAIL  ") + name + ((" | " + str(detail)) if detail else ""))
    assert ok, f"Integrity check failed: {name} {detail}"

need = ["position", "config", "model", "fold", "test_subject", "n_channels", "n_train_windows", "n_test_windows",
        "macro_f1", "macro_f1_workouts", "accuracy", "weighted_f1", "model_size_bytes",
        "latency_total_ms"] + F1COLS
check("required columns present (01)", all(c in res_cls.columns for c in need))
check("required columns present (02)", all(c in res_cnn.columns for c in need + ["seed", "final_train_loss", "n_params"]))
check("models in 01 are RF, SVM, XGB", set(res_cls.model) == {"RF", "SVM", "XGB"}, sorted(set(res_cls.model)))
check("model in 02 is CNN", set(res_cnn.model) == {"CNN"})
for nm, r in [("01", res_cls), ("02", res_cnn)]:
    check(f"positions as expected ({nm})", set(r.position) == set(POS), sorted(set(r.position)))
    check(f"15 configurations as expected ({nm})", set(r.config) == set(CFG_ORDER))
    check(f"folds are 0..9 ({nm})", set(r.fold) == set(range(10)))

SEEDS = sorted(res_cnn.seed.unique())
key = ["position", "config", "fold"]
check("01 has no duplicate units", not res_cls.duplicated(key + ["model"]).any())
check("02 has no duplicate units", not res_cnn.duplicated(key + ["seed"]).any())
check("01 grid complete (3 models x 3 positions x 15 configs x 10 folds)", len(res_cls) == 3 * 3 * 15 * 10, len(res_cls))
check("02 grid complete (3 positions x 15 configs x 10 folds x n_seeds)", len(res_cnn) == 3 * 15 * 10 * len(SEEDS),
      f"{len(res_cnn)} rows, seeds {SEEDS}")

# average CNN seeds per unit (identity when there is one seed)
keep_first = ["test_subject", "n_channels", "channels", "n_features", "n_train_windows", "n_test_windows", "model",
              "epochs", "n_params", "macs_per_window", "data_rate_hz", "model_size_bytes"]
keep_first = [c for c in keep_first if c in res_cnn.columns]
mean_cols = [c for c in res_cnn.columns if res_cnn[c].dtype.kind in "fi" and c not in key + ["seed"] + keep_first]
agg = {c: "first" for c in keep_first}
agg.update({c: "mean" for c in mean_cols})
cnn_unit = res_cnn.groupby(key, as_index=False).agg(agg)
log(f"CNN: {len(res_cnn)} seed-level rows -> {len(cnn_unit)} unit rows (mean over {len(SEEDS)} seed(s))")

df = pd.concat([res_cls.assign(seed=np.nan), cnn_unit], ignore_index=True)
df["data_rate_hz"] = df["n_channels"] * FS
df["n_params_eff"] = df["n_params"].fillna(df["n_params_proxy"]) if "n_params_proxy" in df.columns else df["n_params"]
df["model"] = pd.Categorical(df["model"], MODELS, ordered=True)
df["config"] = pd.Categorical(df["config"], CFG_ORDER, ordered=True)
df["position"] = pd.Categorical(df["position"], POS, ordered=True)
df = df.sort_values(["position", "model", "config", "fold"]).reset_index(drop=True)
check("combined table is 4 models x 3 positions x 15 configs x 10 folds", len(df) == 4 * 3 * 15 * 10, len(df))

check("each fold maps to exactly one held-out subject (all positions and models)",
      df.groupby("fold").test_subject.nunique().max() == 1)
SUBJECTS = sorted(df.test_subject.astype(int).unique())
df["test_subject"] = df["test_subject"].astype(int)
check("10 distinct held-out subjects", len(SUBJECTS) == 10, SUBJECTS)
chk = df.groupby(["position", "fold"], observed=True)[["n_test_windows", "n_train_windows"]].nunique().max().max()
check("same train/test window counts for all models and configs within a (position, fold)", chk == 1)
NCH = df.groupby("config", observed=True).n_channels.first().to_dict()
check("n_channels matches the configuration name",
      all(NCH[c] == (6 if c == "6-axis" else 3 if c in ("acc-3", "gyro-3") else len(re.findall(r"[AG][xyz]", c))) for c in CFG_ORDER), NCH)
vals = df[METRICS + ["accuracy", "weighted_f1"] + F1COLS]
check("metrics are finite and within [0, 1]", np.isfinite(vals.values).all() and (vals.values >= 0).all() and (vals.values <= 1).all())

# notebook 02 control checks
check("label-shuffle control passed in notebook 02", sanity["label_shuffle_macro_f1"] < sanity["threshold"],
      f"shuffled macro-F1 {sanity['label_shuffle_macro_f1']:.4f} (threshold {sanity['threshold']}, chance {sanity['chance_reference']:.4f})")
check("notebook 02 verified its windows against notebook 01", env_cnn.get("windows_verified_against_notebook_01") is True)
check("both notebooks used the same RecGym.csv (SHA-256)", env_cnn["data_sha256"] == env_cls["data_sha256"], env_cls["data_sha256"][:16] + "...")
check("notebook 02 ran with seeds as recorded", sorted(env_cnn["seeds"]) == [int(s) for s in SEEDS], env_cnn["seeds"])
check("notebook 02 was the full run, not the smoke test", env_cnn.get("smoke_test") is False)
try:
    wp = locate(d1, "windows_index.parquet")
    log("Informational: sha256(windows_index.parquet) =", sha256_file(wp)[:16], "| notebook 02 windows hash =",
        env_cnn.get("windows_sha256", "")[:16], "(hashing methods may differ; notebook 02 verified the windows internally)")
    wm = json.load(open(locate(d1, "windows_meta.json")))
    log("Windowing:", {k: wm[k] for k in ["window_samples", "step", "fs"]})
except AssertionError as e:
    log("windows_index.parquet / windows_meta.json not found in the 01 zip (informational only):", e)

# class support per (position, fold) from the confusion matrices; per-class F1 of an absent class is not defined
def support_table(conf, extra):
    return conf.groupby(["position", "fold", "true"] + extra).n.sum().reset_index()
s_cls = support_table(conf_cls, ["model", "config"])
s_cnn = support_table(conf_cnn, ["model", "config", "seed"])
check("class support is identical for every model/config in 01", s_cls.groupby(["position", "fold", "true"]).n.nunique().max() == 1)
check("class support is identical for every config/seed in 02", s_cnn.groupby(["position", "fold", "true"]).n.nunique().max() == 1)
sup_a = s_cls.groupby(["position", "fold", "true"]).n.first()
sup_b = s_cnn.groupby(["position", "fold", "true"]).n.first()
check("class support agrees between 01 and 02", sup_a.reindex(sup_b.index).fillna(-1).equals(sup_b))
pres = pd.DataFrame(0, index=pd.MultiIndex.from_product([POS, range(10)], names=["position", "fold"]), columns=CLASSES)
for (p, f, c), n in sup_a.items():
    if n > 0:
        pres.loc[(p, f), c] = 1
PM = pres.reindex(pd.MultiIndex.from_frame(df[["position", "fold"]].astype({"position": str}))).values
Fm = df[F1COLS].values.astype(float)
Fm[PM == 0] = np.nan
df[F1COLS] = Fm
log(f"Per-class F1 masked to NaN where the class is absent from the held-out subject: {int(np.isnan(Fm).sum())} of {Fm.size} cells")

# consistency of the saved macro metrics with the per-class F1 values (informational)
mf_chk = np.nanmean(Fm, axis=1)
wk_idx = [CLASSES.index(c) for c in WORKOUTS]
mw_chk = np.nanmean(Fm[:, wk_idx], axis=1)
e1, e2 = np.abs(mf_chk - df["macro_f1"].values).max(), np.abs(mw_chk - df["macro_f1_workouts"].values).max()
CHECKS["macro-F1 recomputed from per-class F1 (max abs error)"] = {"ok": bool(max(e1, e2) < 1e-4), "detail": f"all-12: {e1:.2e}; workouts: {e2:.2e}"}
log(("PASS  " if max(e1, e2) < 1e-4 else "WARN  ") + f"macro-F1 recomputed from per-class F1 over present classes: max abs error {e1:.2e} (12 labels), {e2:.2e} (11 workouts)")

log("\nALL HARD INTEGRITY CHECKS PASSED. Rows analysed:", len(df))
display(df.groupby(["position", "model"], observed=True)[PRIMARY_METRIC].agg(["count", "mean", "std"]).round(3))

## 2 · Statistics helpers

In [ ]:
MIN_P = 2 / 2 ** 10
log(f"Smallest possible exact two-sided Wilcoxon p with n = 10 pairs: {MIN_P:.5f}")

def paired_stats(d):
    d = np.asarray(d, float)
    d = d[np.isfinite(d)]
    n = len(d)
    sd = d.std(ddof=1) if n > 1 else np.nan
    nz = d[d != 0]
    if n < 2:
        return dict(n=n, mean_diff=np.nan, median_diff=np.nan, ci_lo=np.nan, ci_hi=np.nan, dz=np.nan, rbc=np.nan,
                    n_pos=int((d > 0).sum()), n_neg=int((d < 0).sum()), n_zero=int((d == 0).sum()), p=np.nan)
    if len(nz) == 0:
        p, rbc = 1.0, 0.0
    else:
        r = rankdata(np.abs(nz))
        wp, wm = r[nz > 0].sum(), r[nz < 0].sum()
        rbc = (wp - wm) / (wp + wm)
        try:
            p = float(wilcoxon(d, zero_method="wilcox", alternative="two-sided", method="auto").pvalue)
        except ValueError:
            p = 1.0
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, n, size=(N_BOOT, n))
    bm = d[idx].mean(axis=1)
    lo, hi = np.percentile(bm, [2.5, 97.5])
    return dict(n=n, mean_diff=float(d.mean()), median_diff=float(np.median(d)), ci_lo=float(lo), ci_hi=float(hi),
                dz=float(d.mean() / sd) if sd and sd > 0 else np.nan, rbc=float(rbc),
                n_pos=int((d > 0).sum()), n_neg=int((d < 0).sum()), n_zero=int((d == 0).sum()), p=p)

def holm(p):
    p = np.asarray(p, float)
    out = np.full(len(p), np.nan)
    ok = np.isfinite(p)
    pv = p[ok]
    m = len(pv)
    order = np.argsort(pv)
    adj = np.empty(m)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * pv[i])
        adj[i] = min(1.0, run)
    out[ok] = adj
    return out

def add_holm(t, fam_col):
    t = t.copy()
    t["p_holm_family"] = np.nan
    for _, idx in t.groupby(fam_col).groups.items():
        t.loc[idx, "p_holm_family"] = holm(t.loc[idx, "p"].values)
    t["p_holm_global"] = holm(t["p"].values)
    t["sig_family"] = t["p_holm_family"] < ALPHA
    return t

def build_V(metric):
    V = df.pivot_table(index=["position", "model", "config"], columns="test_subject", values=metric,
                       aggfunc="mean", observed=True)
    V = V.reindex(columns=SUBJECTS)
    assert V.shape == (3 * 4 * 15, 10) and not V.isna().any().any(), "incomplete subject x unit matrix"
    return V

def vec(V, pos, model, cfg):
    return V.loc[(pos, model, cfg)].values.astype(float)

def pooled(V, pos, model):
    return V.loc[(pos, model)].mean(axis=0).values.astype(float)

def cfg_axes(name):
    if name == "6-axis": return frozenset(IMU)
    if name == "acc-3": return frozenset(["Ax", "Ay", "Az"])
    if name == "gyro-3": return frozenset(["Gx", "Gy", "Gz"])
    return frozenset(re.findall(r"[AG][xyz]", name))

AX2CFG = {cfg_axes(c): c for c in CFG_ORDER}
assert len(AX2CFG) == 15
ACC_PAIRS = [c for c in CFG_ORDER if c.startswith("acc-A") and c != "acc-3"]
GYR_PAIRS = [c for c in CFG_ORDER if c.startswith("gyro-G") and c != "gyro-3"]
ACC_SING = ["single-Ax", "single-Ay", "single-Az"]
GYR_SING = ["single-Gx", "single-Gy", "single-Gz"]
LEVELS = {"triple": (["acc-3"], ["gyro-3"]), "pair": (ACC_PAIRS, GYR_PAIRS), "single": (ACC_SING, GYR_SING)}
GROUPS = {"acc": ["Ax", "Ay", "Az"], "gyro": ["Gx", "Gy", "Gz"]}
POS_PAIRS = [("wrist", "leg"), ("wrist", "pocket"), ("leg", "pocket")]
MODEL_PAIRS = list(itertools.combinations(MODELS, 2))

## 3 · Inference blocks A to E (run on the primary metric and on the Null-excluded metric)

In [ ]:
def run_inference(metric):
    V = build_V(metric)
    T = {}

    # ---- A: each reduced configuration vs 6-axis ----
    rows = []
    for pos in POS:
        for m in MODELS:
            ref = vec(V, pos, m, "6-axis")
            for cfg in CFG_ORDER[1:]:
                x = vec(V, pos, m, cfg)
                rows.append(dict(metric=metric, position=pos, model=m, config=cfg, n_channels=NCH[cfg],
                                 mean_6axis=ref.mean(), mean_cfg=x.mean(), sd_cfg=x.std(ddof=1),
                                 retained=x.mean() / ref.mean(), **paired_stats(x - ref)))
    A = pd.DataFrame(rows)
    A["fam"] = A.position + "|" + A.model
    T["A_reduction_vs_6axis"] = add_holm(A, "fam")

    # ---- B1 / B2: accelerometer minus gyroscope gap, and its dependence on position ----
    gap = {}
    rows = []
    for pos in POS:
        for m in MODELS:
            for lev, (ac, gy) in LEVELS.items():
                g = np.mean([vec(V, pos, m, c) for c in ac], axis=0) - np.mean([vec(V, pos, m, c) for c in gy], axis=0)
                gap[(pos, m, lev)] = g
                rows.append(dict(metric=metric, position=pos, model=m, level=lev, **paired_stats(g)))
    B1 = pd.DataFrame(rows)
    B1["fam"] = B1.model
    T["B1_acc_minus_gyro_gap"] = add_holm(B1, "fam")
    rows = []
    for m in MODELS:
        for lev in LEVELS:
            for p1, p2 in POS_PAIRS:
                rows.append(dict(metric=metric, model=m, level=lev, contrast=f"{p1} - {p2}",
                                 **paired_stats(gap[(p1, m, lev)] - gap[(p2, m, lev)])))
    B2 = pd.DataFrame(rows)
    B2["fam"] = B2.model
    T["B2_gap_difference_between_positions"] = add_holm(B2, "fam")

    # ---- C: marginal gain of an axis within its own sensor group ----
    rows = []
    for pos in POS:
        for m in MODELS:
            for grp, axes in GROUPS.items():
                for a in axes:
                    others = [b for b in axes if b != a]
                    gains = []
                    for r in (1, 2):
                        for S in itertools.combinations(others, r):
                            gains.append(vec(V, pos, m, AX2CFG[frozenset(S) | {a}]) - vec(V, pos, m, AX2CFG[frozenset(S)]))
                    rows.append(dict(metric=metric, position=pos, model=m, modality=grp, axis=a, n_subsets=len(gains),
                                     **paired_stats(np.mean(gains, axis=0))))
    C = pd.DataFrame(rows)
    C["fam"] = C.model
    T["C_axis_marginal_gain"] = add_holm(C, "fam")

    # ---- D: model comparison ----
    rows = []
    for pos in POS:
        for cfg in KEY_CONFIGS + ["pooled_15_configs"]:
            for m1, m2 in MODEL_PAIRS:
                if cfg == "pooled_15_configs":
                    d = pooled(V, pos, m1) - pooled(V, pos, m2)
                else:
                    d = vec(V, pos, m1, cfg) - vec(V, pos, m2, cfg)
                rows.append(dict(metric=metric, position=pos, config=cfg, contrast=f"{m1} - {m2}", **paired_stats(d)))
    D = pd.DataFrame(rows)
    D["fam"] = D.position + "|" + D.config
    T["D_model_comparison"] = add_holm(D, "fam")

    # ---- E: position comparison (same subjects at every position) ----
    rows = []
    for m in MODELS:
        for cfg in KEY_CONFIGS + ["pooled_15_configs"]:
            for p1, p2 in POS_PAIRS:
                if cfg == "pooled_15_configs":
                    d = pooled(V, p1, m) - pooled(V, p2, m)
                else:
                    d = vec(V, p1, m, cfg) - vec(V, p2, m, cfg)
                rows.append(dict(metric=metric, model=m, config=cfg, contrast=f"{p1} - {p2}", **paired_stats(d)))
    E = pd.DataFrame(rows)
    E["fam"] = E.model + "|" + E.config
    T["E_position_comparison"] = add_holm(E, "fam")
    return T, V

TABLES, VMAT = {}, {}
for met in METRICS:
    TABLES[met], VMAT[met] = run_inference(met)
    tag = "primary" if met == PRIMARY_METRIC else "sens_workouts_only"
    for name, t in TABLES[met].items():
        save_table(t, f"{name}__{tag}")

A1 = TABLES[PRIMARY_METRIC]["A_reduction_vs_6axis"]
log("\nBlock A (primary metric): reduced configs significantly below 6-axis after Holm within position x model (of 14):")
display(A1.assign(sig_below=A1.sig_family & (A1.mean_diff < 0)).groupby(["position", "model"])["sig_below"].sum().unstack())
log("Remember: with n = 10 the exact Wilcoxon cannot go below p = 0.00195, so read effect sizes and sign counts together with p.")

In [ ]:
# ---- concordance between the primary and the Null-excluded metric (block A, B1) ----
rows = []
for name in ["A_reduction_vs_6axis", "B1_acc_minus_gyro_gap", "C_axis_marginal_gain"]:
    a, b = TABLES[PRIMARY_METRIC][name], TABLES[SENS_METRIC][name]
    ok = a.mean_diff.values, b.mean_diff.values
    rho = spearmanr(ok[0], ok[1]).statistic
    agree_sign = float((np.sign(ok[0]) == np.sign(ok[1])).mean())
    agree_sig = float((a.sig_family.values == b.sig_family.values).mean())
    rows.append(dict(block=name, n_rows=len(a), spearman_mean_diff=rho, same_sign_share=agree_sign, same_family_significance_share=agree_sig))
CONC = save_table(pd.DataFrame(rows), "null_sensitivity_concordance")
display(CONC.round(3))

## 4 · Descriptive tables: means, rankings, per-subject results

In [ ]:
Vp = VMAT[PRIMARY_METRIC]

# mean and sd over the 10 held-out subjects, both metrics, plus accuracy
long_rows = []
for (pos, m, cfg), g in df.groupby(["position", "model", "config"], observed=True):
    long_rows.append(dict(position=pos, model=m, config=cfg, n_channels=NCH[cfg], data_rate_hz=NCH[cfg] * FS,
                          macro_f1_mean=g.macro_f1.mean(), macro_f1_sd=g.macro_f1.std(ddof=1),
                          macro_f1_workouts_mean=g.macro_f1_workouts.mean(), macro_f1_workouts_sd=g.macro_f1_workouts.std(ddof=1),
                          accuracy_mean=g.accuracy.mean(), accuracy_sd=g.accuracy.std(ddof=1),
                          weighted_f1_mean=g.weighted_f1.mean(), n_folds=len(g)))
DESC = save_table(pd.DataFrame(long_rows), "descriptive_means_by_position_model_config")

# single-axis ranking per position and model, and rank agreement between positions (descriptive only; 6 items)
rows, rk = [], {}
for m in MODELS:
    for pos in POS:
        s = pd.Series({c: vec(Vp, pos, m, c).mean() for c in ACC_SING + GYR_SING})
        rk[(m, pos)] = s
        for c, v in s.items():
            rows.append(dict(model=m, position=pos, single_axis=c, mean_macro_f1=v, rank=int(s.rank(ascending=False)[c])))
SINGLE_RANK = save_table(pd.DataFrame(rows), "single_axis_ranking")
rows = []
for m in MODELS:
    for p1, p2 in POS_PAIRS:
        tau = kendalltau(rk[(m, p1)].values, rk[(m, p2)].values).statistic
        rows.append(dict(model=m, position_a=p1, position_b=p2, kendall_tau_of_6_single_axis_means=tau))
save_table(pd.DataFrame(rows), "single_axis_rank_agreement_between_positions")

# per-subject results
ps = df[["position", "model", "config", "test_subject", "macro_f1", "macro_f1_workouts", "accuracy"]].copy()
save_table(ps, "per_subject_results_long")
rows = []
for pos in POS:
    for m in MODELS:
        ref = vec(Vp, pos, m, "6-axis")
        for cfg in KEY_CONFIGS[1:]:
            x = vec(Vp, pos, m, cfg)
            for s, a, b in zip(SUBJECTS, ref, x):
                rows.append(dict(position=pos, model=m, config=cfg, subject=s, macro_f1_6axis=a, macro_f1_cfg=b, diff=b - a))
save_table(pd.DataFrame(rows), "per_subject_reduction_effect_key_configs")
subj_rng = (df[(df.config == "6-axis")].groupby(["position", "model"], observed=True).macro_f1
            .agg(min="min", max="max", sd="std").reset_index())
save_table(subj_rng, "per_subject_spread_6axis")
display(subj_rng.round(3))

## 5 · RQ2: performance versus cost (channels, data rate, model size, latency)

Efficiency is argued with **channel count and data rate (channels × 20 Hz)** and with measured latency. CNN parameter count is almost identical across configurations, so it is not an efficiency axis for the CNN.

In [ ]:
g = df.groupby(["position", "model", "config"], observed=True)
COST = g.agg(n_channels=("n_channels", "first"), data_rate_hz=("data_rate_hz", "first"),
             macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
             macro_f1_workouts_mean=("macro_f1_workouts", "mean"), accuracy_mean=("accuracy", "mean"),
             model_size_bytes_median=("model_size_bytes", "median"), n_params_median=("n_params_eff", "median"),
             macs_per_window_median=("macs_per_window", "median"),
             latency_total_ms_median=("latency_total_ms", "median"),
             latency_total_ms_iqr=("latency_total_ms", lambda s: s.quantile(.75) - s.quantile(.25)),
             train_time_s_median=("train_time_s", "median")).reset_index()

# Pareto flag: not dominated in (fewer channels, higher mean macro-F1), within each position x model
flags = []
for (pos, m), gg in COST.groupby(["position", "model"], observed=True):
    for i, r in gg.iterrows():
        dom = ((gg.n_channels <= r.n_channels) & (gg.macro_f1_mean >= r.macro_f1_mean) &
               ((gg.n_channels < r.n_channels) | (gg.macro_f1_mean > r.macro_f1_mean))).any()
        flags.append((i, not dom))
COST["pareto_channels_vs_f1"] = pd.Series(dict(flags))
ref6 = COST[COST.config == "6-axis"].set_index(["position", "model"]).macro_f1_mean
COST["retained_vs_6axis"] = [r.macro_f1_mean / ref6[(r.position, r.model)] for r in COST.itertuples()]
save_table(COST, "rq2_cost_table")

# descriptive non-inferiority: smallest channel count whose paired-bootstrap 95% CI lower bound for (config - 6-axis) exceeds -delta
rows = []
for delta in DELTAS:
    for pos in POS:
        for m in MODELS:
            a = A1[(A1.position == pos) & (A1.model == m)]
            ok = a[a.ci_lo > -delta]
            if len(ok) == 0:
                rows.append(dict(delta=delta, position=pos, model=m, min_channels=6, configs="6-axis only (no reduced config qualifies)",
                                 mean_diffs=""))
            else:
                k = ok.n_channels.min()
                sel = ok[ok.n_channels == k]
                rows.append(dict(delta=delta, position=pos, model=m, min_channels=int(k),
                                 configs="; ".join(sel.config), mean_diffs="; ".join(f"{v:+.3f}" for v in sel.mean_diff)))
NONINF = save_table(pd.DataFrame(rows), "rq2_noninferiority_descriptive")
display(NONINF)
log("Non-inferiority here is descriptive: bootstrap CIs over 10 subjects are crude, and the margins are not clinically derived.")

# latency summary with explicit hardware note
LAT_NOTE = (f"CNN latency: batch 1, 1 CPU thread, {env_cnn['cpu_for_latency']} ({env_cnn['cpu_count']} cores), includes slicing, z-score and forward pass. "
            f"Classical latency: notebook 01, CPU model not recorded, includes feature extraction and predict. Colab CPUs vary between sessions; not edge-device performance.")
log(LAT_NOTE)
lat = (COST[COST.config.isin(["6-axis", "single-Ax"])]
       .pivot_table(index=["model", "config"], columns="position", values="latency_total_ms_median", observed=True))
display(lat.round(3))
open(OUT / "checks" / "latency_note.txt", "w").write(LAT_NOTE)

## 6 · RQ4: per-class robustness and pooled confusions

In [ ]:
pc = df.melt(id_vars=["position", "model", "config", "fold", "test_subject"], value_vars=F1COLS,
             var_name="class", value_name="f1")
pc["class"] = pc["class"].str[3:]
pc["position"] = pc["position"].astype(str); pc["model"] = pc["model"].astype(str); pc["config"] = pc["config"].astype(str)

cls_mean = (pc.dropna(subset=["f1"]).groupby(["position", "model", "config", "class"])
            .f1.agg(f1_mean="mean", f1_sd="std", n_folds_present="count").reset_index())
save_table(cls_mean, "perclass_f1_mean_by_position_model_config")

PC = pc.pivot_table(index=["position", "model", "class", "config"], columns="test_subject", values="f1", aggfunc="mean")
PC = PC.reindex(columns=SUBJECTS)

# mean paired drop vs 6-axis for every configuration (descriptive), tests only for acc-3 and gyro-3
rows = []
for pos in POS:
    for m in MODELS:
        for c in CLASSES:
            ref = PC.loc[(pos, m, c, "6-axis")].values.astype(float)
            for cfg in CFG_ORDER[1:]:
                x = PC.loc[(pos, m, c, cfg)].values.astype(float)
                d = (x - ref)[np.isfinite(x) & np.isfinite(ref)]
                row = dict(position=pos, model=m, **{"class": c}, config=cfg, n_folds_present=len(d),
                           mean_f1_6axis=np.nanmean(ref) if np.isfinite(ref).any() else np.nan,
                           mean_drop=d.mean() if len(d) else np.nan)
                if cfg in ("acc-3", "gyro-3") and len(d) >= 6:
                    st = paired_stats(d)
                    row.update(ci_lo=st["ci_lo"], ci_hi=st["ci_hi"], dz=st["dz"], rbc=st["rbc"], n_pos=st["n_pos"], n_neg=st["n_neg"], p=st["p"])
                rows.append(row)
CLS_DROP = pd.DataFrame(rows)
tested = CLS_DROP.p.notna()
CLS_DROP["fam"] = CLS_DROP.position + "|" + CLS_DROP.model
CLS_DROP["p_holm_family"] = np.nan
for _, idx in CLS_DROP[tested].groupby("fam").groups.items():
    CLS_DROP.loc[idx, "p_holm_family"] = holm(CLS_DROP.loc[idx, "p"].values)
CLS_DROP["sig_family"] = CLS_DROP.p_holm_family < ALPHA
save_table(CLS_DROP, "perclass_drop_vs_6axis")

rob = (CLS_DROP.groupby(["position", "model", "class"]).mean_drop.mean().reset_index()
       .rename(columns={"mean_drop": "mean_drop_over_14_reduced_configs"}))
save_table(rob, "perclass_robustness_summary")

# pooled confusion matrices (pooled windows over the 10 folds; subjects weighted by window count)
conf_all = pd.concat([conf_cls[["position", "config", "model", "true", "pred", "n"]],
                      conf_cnn[["position", "config", "model", "true", "pred", "n"]]], ignore_index=True)
pooled_conf = conf_all.groupby(["position", "model", "config", "true", "pred"], as_index=False).n.sum()
save_table(pooled_conf, "pooled_confusion_long")

def cm_matrix(pos, model, cfg, normalize=True):
    t = pooled_conf[(pooled_conf.position == pos) & (pooled_conf.model == model) & (pooled_conf.config == cfg)]
    M = t.pivot_table(index="true", columns="pred", values="n", aggfunc="sum").reindex(index=CLASSES, columns=CLASSES).fillna(0).values
    if normalize:
        rs = M.sum(axis=1, keepdims=True)
        M = np.divide(M, rs, out=np.zeros_like(M, dtype=float), where=rs > 0)
    return M

rows = []
for pos in POS:
    for m in MODELS:
        for cfg in KEY_CONFIGS:
            M = cm_matrix(pos, m, cfg)
            Mraw = cm_matrix(pos, m, cfg, normalize=False)
            off = [(M[i, j], Mraw[i, j], CLASSES[i], CLASSES[j]) for i in range(12) for j in range(12) if i != j and Mraw[i, j] > 0]
            for rate, n, t_, p_ in sorted(off, reverse=True)[:6]:
                rows.append(dict(position=pos, model=m, config=cfg, true_class=t_, predicted_as=p_, share_of_true_class=rate, windows=int(n)))
save_table(pd.DataFrame(rows), "top_confusions_key_configs")

## 7 · Figures

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

def heat(ax, M, rlabels, clabels, title="", vmin=None, vmax=None, fmt="{:.2f}", cmap="viridis", annotate=True, fs=7):
    im = ax.imshow(M, aspect="auto", vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xticks(range(len(clabels))); ax.set_xticklabels(clabels, rotation=60, ha="right", fontsize=fs + 1)
    ax.set_yticks(range(len(rlabels))); ax.set_yticklabels(rlabels, fontsize=fs + 1)
    ax.set_title(title, fontsize=9)
    if annotate:
        lo, hi = np.nanmin(M), np.nanmax(M); thr = (lo + hi) / 2
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                v = M[i, j]
                if np.isfinite(v):
                    ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=fs, color="white" if (v < thr and cmap == "viridis") else "black")
    return im

def model_legend(ax, **kw):
    ax.legend(handles=[Line2D([0], [0], marker="o", color=COL[m], lw=0, label=m) for m in MODELS], frameon=False, **kw)

# F1: mean macro-F1, configuration x (position, model)
cols = [(p, m) for p in POS for m in MODELS]
M = np.array([[vec(Vp, p, m, c).mean() for (p, m) in cols] for c in CFG_ORDER])
fig, ax = plt.subplots(figsize=(13, 6.5))
im = heat(ax, M, CFG_ORDER, [f"{p}\n{m}" for p, m in cols], "Mean macro-F1 over 10 held-out subjects (primary metric)", vmin=0.2, vmax=0.95, fs=7)
fig.colorbar(im, ax=ax, shrink=0.8)
savefig(fig, "F1_macroF1_heatmap")

# F2: cost of reduction vs 6-axis with bootstrap CI, filled = Holm-significant within family
fig, axes = plt.subplots(1, 3, figsize=(19, 5), sharey=True)
cfgs = CFG_ORDER[1:]
for ax, pos in zip(axes, POS):
    for k, m in enumerate(MODELS):
        s = A1[(A1.position == pos) & (A1.model == m)].set_index("config").loc[cfgs]
        x = np.arange(len(cfgs)) + (k - 1.5) * 0.15
        ax.errorbar(x, s.mean_diff.values, yerr=[(s.mean_diff - s.ci_lo).values, (s.ci_hi - s.mean_diff).values],
                    fmt="none", ecolor=COL[m], alpha=0.55, lw=1)
        sig = s.sig_family.values
        ax.scatter(x[sig], s.mean_diff.values[sig], color=COL[m], s=26, zorder=3)
        ax.scatter(x[~sig], s.mean_diff.values[~sig], facecolors="white", edgecolors=COL[m], s=26, zorder=3)
    ax.axhline(0, color="k", lw=0.8)
    for dlt in DELTAS[1:2]:
        ax.axhline(-dlt, color="grey", ls=":", lw=0.8)
    ax.set_xticks(range(len(cfgs))); ax.set_xticklabels(cfgs, rotation=60, ha="right")
    ax.set_title(pos)
axes[0].set_ylabel("Mean paired difference in macro-F1 vs 6-axis")
model_legend(axes[0], loc="lower left")
fig.suptitle("Cost of channel reduction (filled = Holm-significant within position x model; bars = bootstrap 95% CI over subjects; dotted = -0.03)", y=1.02)
savefig(fig, "F2_reduction_vs_6axis")

# F3: accelerometer-minus-gyroscope gap by position
B1 = TABLES[PRIMARY_METRIC]["B1_acc_minus_gyro_gap"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, lev in zip(axes, LEVELS):
    for k, m in enumerate(MODELS):
        s = B1[(B1.level == lev) & (B1.model == m)].set_index("position").loc[POS]
        x = np.arange(3) + (k - 1.5) * 0.18
        ax.bar(x, s.mean_diff.values, width=0.17, color=COL[m], alpha=0.85)
        ax.errorbar(x, s.mean_diff.values, yerr=[(s.mean_diff - s.ci_lo).values, (s.ci_hi - s.mean_diff).values], fmt="none", ecolor="k", lw=0.9)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(3)); ax.set_xticklabels(POS); ax.set_title(f"{lev}s: accelerometer minus gyroscope")
axes[0].set_ylabel("Macro-F1 gap (device-frame axes)")
model_legend(axes[0], loc="best")
savefig(fig, "F3_acc_minus_gyro_gap")

# F4: marginal gain of each axis within its own sensor group
C1 = TABLES[PRIMARY_METRIC]["C_axis_marginal_gain"]
AXES6 = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for ax, pos in zip(axes, POS):
    for k, m in enumerate(MODELS):
        s = C1[(C1.position == pos) & (C1.model == m)].set_index("axis").loc[AXES6]
        x = np.arange(6) + (k - 1.5) * 0.15
        ax.errorbar(x, s.mean_diff.values, yerr=[(s.mean_diff - s.ci_lo).values, (s.ci_hi - s.mean_diff).values], fmt="none", ecolor=COL[m], alpha=0.55)
        sig = s.sig_family.values
        ax.scatter(x[sig], s.mean_diff.values[sig], color=COL[m], s=26, zorder=3)
        ax.scatter(x[~sig], s.mean_diff.values[~sig], facecolors="white", edgecolors=COL[m], s=26, zorder=3)
    ax.axhline(0, color="k", lw=0.8); ax.set_xticks(range(6)); ax.set_xticklabels(AXES6); ax.set_title(pos)
axes[0].set_ylabel("Mean marginal gain in macro-F1")
model_legend(axes[0], loc="best")
fig.suptitle("Marginal gain of adding an axis to a non-empty subset of the same sensor (device frame; filled = Holm-significant within model)", y=1.02)
savefig(fig, "F4_axis_marginal_gain")

# F5: performance versus channels
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), sharey=True)
for ax, pos in zip(axes, POS):
    for m in MODELS:
        s = COST[(COST.position == pos) & (COST.model == m)]
        jit = (MODELS.index(m) - 1.5) * 0.06
        ax.scatter(s.n_channels + jit, s.macro_f1_mean, color=COL[m], s=22, alpha=0.7)
        best = s.groupby("n_channels").macro_f1_mean.max()
        ax.plot(best.index + jit, best.values, color=COL[m], lw=1)
    ax.set_xticks([1, 2, 3, 6]); ax.set_xlabel("Channels (data rate = channels x 20 Hz)"); ax.set_title(pos)
axes[0].set_ylabel("Mean macro-F1")
model_legend(axes[0], loc="lower right")
fig.suptitle("Performance versus channel count (points = configurations; line = best configuration per channel count)", y=1.02)
savefig(fig, "F5_performance_vs_channels")

# F6: latency versus macro-F1
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for ax, pos in zip(axes, POS):
    for m in MODELS:
        s = COST[(COST.position == pos) & (COST.model == m)]
        ax.scatter(s.latency_total_ms_median, s.macro_f1_mean, color=COL[m], s=22, alpha=0.75)
    ax.set_xscale("log"); ax.set_xlabel("Single-window CPU latency, median (ms, log)"); ax.set_title(pos)
axes[0].set_ylabel("Mean macro-F1")
model_legend(axes[0], loc="lower right")
fig.suptitle("Latency vs performance. Hardware differs between 01 and 02 (CPU of 01 not recorded). Not edge-device latency.", y=1.02)
savefig(fig, "F6_latency_vs_f1")

# F7: per-class F1 heatmaps
for m in FIG_MODELS:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
    for ax, pos in zip(axes, POS):
        sub = cls_mean[(cls_mean.position == pos) & (cls_mean.model == m)].pivot(index="class", columns="config", values="f1_mean")
        Mc = sub.reindex(index=CLASSES, columns=CFG_ORDER).values
        im = heat(ax, Mc, CLASSES, CFG_ORDER, f"{m} - {pos}", vmin=0, vmax=1, fs=6)
    fig.colorbar(im, ax=axes, shrink=0.8)
    savefig(fig, f"F7_perclass_f1_{m}")

# F8: pooled confusion matrices (row-normalized)
for m in FIG_MODELS:
    fig, axes = plt.subplots(3, 3, figsize=(15, 14))
    for i, pos in enumerate(POS):
        for j, cfg in enumerate(KEY_CONFIGS):
            ax = axes[i, j]
            ax.imshow(cm_matrix(pos, m, cfg), vmin=0, vmax=1, cmap="Blues")
            ax.set_xticks(range(12)); ax.set_yticks(range(12))
            ax.set_xticklabels(CLASSES if i == 2 else [], rotation=90, fontsize=6)
            ax.set_yticklabels(CLASSES if j == 0 else [], fontsize=6)
            ax.set_title(f"{m} | {pos} | {cfg}", fontsize=8)
    savefig(fig, f"F8_confusion_{m}")

# F9: per-subject heatmap for 6-axis, and per-subject effect of dropping to acc-3 / gyro-3
cols = [(p, m) for p in POS for m in MODELS]
Ms = np.array([vec(Vp, p, m, "6-axis") for (p, m) in cols]).T
fig, axes = plt.subplots(1, 3, figsize=(20, 4.6))
heat(axes[0], Ms, [f"S{s}" for s in SUBJECTS], [f"{p}\n{m}" for p, m in cols], "6-axis macro-F1 per held-out subject", vmin=0.2, vmax=1, fs=6)
for ax, cfg in zip(axes[1:], ["acc-3", "gyro-3"]):
    Md = np.array([vec(Vp, p, m, cfg) - vec(Vp, p, m, "6-axis") for (p, m) in cols]).T
    heat(ax, Md, [f"S{s}" for s in SUBJECTS], [f"{p}\n{m}" for p, m in cols], f"{cfg} minus 6-axis", vmin=-0.4, vmax=0.1, cmap="coolwarm", fs=6)
savefig(fig, "F9_per_subject")

## 8 · CNN diagnostics (descriptive, no tuning on test-fold scores)

If the CNN visibly underfits for some configurations, do **not** tune against these scores. A separate nested subject-grouped tuning step would have to be added and labelled as such.

In [ ]:
loss_tab = (res_cnn.groupby(["position", "config"]).final_train_loss.agg(mean="mean", sd="std", max="max").reset_index())
loss_tab["config"] = pd.Categorical(loss_tab.config, CFG_ORDER, ordered=True)
loss_tab = loss_tab.sort_values(["position", "config"])
save_table(loss_tab, "cnn_final_train_loss")
display(loss_tab.pivot(index="config", columns="position", values="mean").round(3))
if len(SEEDS) > 1:
    sn = (res_cnn.groupby(["position", "config", "fold"]).macro_f1.std(ddof=1).groupby(["position", "config"]).mean()
          .reset_index(name="mean_seed_sd_macro_f1"))
    save_table(sn, "cnn_seed_noise")
    display(sn.groupby("position").mean_seed_sd_macro_f1.describe().round(4))
else:
    log("Only one CNN seed (42): training noise is NOT quantified. Add seeds in notebook 02 (SEEDS list) if the CNN claims matter.")

## 9 · Manifest, log, and zip

In [ ]:
manifest = {
    "notebook": "03_analysis.ipynb",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "inputs": {ZIP_01: sha256_file(zip1), ZIP_02: sha256_file(zip2),
               "01_results.csv": sha256_file(locate(d1, "results.csv")), "02_results.csv": sha256_file(locate(d2, "results.csv"))},
    "parameters": {"SEED": SEED, "N_BOOT": N_BOOT, "ALPHA": ALPHA, "DELTAS": DELTAS, "PRIMARY_METRIC": PRIMARY_METRIC,
                   "SENS_METRIC": SENS_METRIC, "KEY_CONFIGS": KEY_CONFIGS, "FS": FS},
    "cnn_seeds": [int(s) for s in SEEDS],
    "data_sha256": env_cls["data_sha256"],
    "classical_config_hash": env_cls["config_hash"], "cnn_config_hash": env_cnn["config_hash"],
    "integrity_checks": CHECKS,
    "packages": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                 "scipy": scipy.__version__, "matplotlib": matplotlib.__version__},
    "notes": ["Wilcoxon min possible exact p with n=10 is 2/1024", "Axes are device-frame",
              "Latency: Colab CPU, not edge", "Provider normalization undocumented",
              "CNN has one seed unless cnn_seeds lists more"],
}
json.dump(manifest, open(OUT / "checks" / "analysis_manifest.json", "w"), indent=2)
json.dump(CHECKS, open(OUT / "checks" / "integrity_checks.json", "w"), indent=2)
open(OUT / "analysis_log.txt", "w").write("\n".join(LOG))

zpath = shutil.make_archive("/content/recgym_03_analysis" if IN_COLAB else "recgym_03_analysis", "zip", OUT)
log("Zip written:", zpath)
if IN_COLAB:
    dest = Path(RESULTS_DIR) / "03_analysis"
    try:
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy(zpath, dest / Path(zpath).name)
        log("Copied to Drive:", dest / Path(zpath).name)
    except Exception as e:
        log("Drive copy skipped:", e)
    from google.colab import files
    files.download(zpath)